# Local-magnitude calculation for the Ulsan-Fault blast-clean catalog

**Goal.** End-to-end ML chain for every event in `catalog_phasenet_plus_2010_2024_blastclean.csv`:
raw counts → displacement (m) via instrument-response deconvolution → Wood-Anderson simulation
(mm) → peak amplitude → magnitude via a region-specific attenuation formula. Ships with
**Sheen et al. 2018** (South Korea, BSSA 108(5A)) as the working baseline; one-line swap-in
point for Heo et al. 2024 when you implement it.

**Self-contained.** All inputs except the catalog and the per-event waveforms live under
[local_magnitudes/](.). The master StationXML was copied into `responses/master/` at setup
(from the now-removable `5.P_wave_moment_tensor_inversion/Response/`). Per-station XML for
stations missing from the master goes into `responses/fetched/`.

**Benchmark.** §5 computes ML for three well-known KMA events (2016 Gyeongju M5.8 + M5.4 +
M4.5 aftershock) and prints residuals vs the KMA-published values. Use this to calibrate
trust in the chain before computing ML for ~15k small events.

## 1. Setup + coverage check

Load the merged inventory (master + fetched) and confirm every station in the continuous
archive has response information available. Missing stations are listed explicitly so you
can decide whether to fetch them from NECIS (see the `Claude/fetch_responses.py` helper)
or accept that those stations won't contribute to the ML.

In [ ]:
import os, sys, glob, warnings
sys.path.insert(0, os.path.abspath("."))
import ml_pipeline as mlp
import numpy as np, pandas as pd, matplotlib.pyplot as plt, obspy
warnings.filterwarnings("ignore")     # ObsPy 'sensitivity differs by 5%' chatter, harmless

# --- attenuation formula (one-line swap point for Heo et al. 2024) -------
ATT_FN = mlp.ml_heo2024          # Heo et al. 2024 (calibrated on GHBSN, southeastern Korea)
# ATT_FN = mlp.ml_sheen2018     # legacy: broader-South-Korea calibration

# --- input paths ---------------------------------------------------------
HERE = os.path.abspath(".")
MASTER_XML  = os.path.join(HERE, "responses/master")
FETCHED_XML = os.path.join(HERE, "responses/fetched")
EVENT_WF_ROOT = "/home/msseo/works/02.Ulsan_Fault_detection/KS_KG/HypoInv/event_waveforms_blastclean"
CONT_ROOT   = "/home/msseo/works/02.Ulsan_Fault_detection/KS_KG/continuous"

# --- load merged inventory ----------------------------------------------
inv = mlp.load_combined_inventory(MASTER_XML, FETCHED_XML)
n_sta = sum(len(n) for n in inv)
print(f"inventory: {n_sta} stations across {len(inv)} network(s)")

# --- station roster from the continuous archive ------------------------
archive = sorted(os.listdir(CONT_ROOT))
needed = []
for s in archive:
    samples = glob.glob(f"{CONT_ROOT}/{s}/*.D/*")[:1]
    if samples:
        net = os.path.basename(samples[0]).split(".")[0]
        needed.append((net, s))
print(f"continuous archive: {len(needed)} stations")

cov = mlp.report_coverage(needed, inv)
print(f"\ncoverage: {int(cov.covered.sum())} / {len(cov)} stations covered")
missing = cov[~cov.covered]
if len(missing):
    print("missing (re-run with Claude/fetch_responses.py to add):")
    for r in missing.itertuples():
        print(f"  {r.network}.{r.station}")
else:
    print("all stations covered")

## 2. One-trace walkthrough — counts → displacement → Wood-Anderson

Pick the Gyeongju M5.8 mainshock and one nearby station; walk through the chain on a single
trace with intermediate plots. This is the visual sanity check that response removal works:
the displacement trace should be a clean, time-symmetric pulse of mm-scale amplitude (not
blown up by a low-frequency drift, which is the failure mode of an under-stabilised
deconvolution).

**Chain**: `detrend(demean)` → `taper(5%)` → `remove_response(output='DISP', pre_filt=(0.001,
0.005, 50, 100), water_level=60)` → `detrend(demean)` → `taper(5%)` → `simulate(WA)` →
`max(|data|)·1000` (m → mm).

In [ ]:
SAMPLE_EID = "20160912113254"        # Gyeongju M5.8
STATION    = "BBK"                    # 30 km from epicentre
CHANNEL    = "ELZ"                    # vertical, short-period

evdir = os.path.join(EVENT_WF_ROOT, SAMPLE_EID)
sac_path = glob.glob(os.path.join(evdir, f"*.{STATION}.{CHANNEL}.sac"))[0]
print(f"sample: {os.path.basename(sac_path)}")
raw = obspy.read(sac_path)[0]
print(f"raw: {raw.id}  {raw.stats.starttime} → {raw.stats.endtime}  {raw.stats.npts} samp @ {raw.stats.sampling_rate} Hz")

# Pipeline on a 1-trace Stream
st = obspy.Stream([raw])
disp = mlp.remove_response_to_disp(st, inv)
wa = disp.copy().simulate(paz_simulate=mlp.WOOD_ANDERSON_PAZ)

# Plot all three stages
fig, axes = plt.subplots(3, 1, figsize=(10, 6), dpi=110, sharex=True)
for ax, tr, lab in zip(axes, [raw, disp[0], wa[0]],
                        ["raw counts", "displacement (m)", "Wood-Anderson (m, ×1000 → mm)"]):
    times = tr.times()
    ax.plot(times, tr.data, color="k", lw=0.6)
    peak_idx = int(np.argmax(np.abs(tr.data)))
    ax.plot(times[peak_idx], tr.data[peak_idx], "o", color="red", ms=4,
            label=f"peak |{tr.data[peak_idx]:.3g}|")
    ax.set_ylabel(lab); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="upper right")
axes[-1].set_xlabel("time relative to trace start (s)")
fig.suptitle(f"{SAMPLE_EID}  {raw.id}  —  raw → displacement → Wood-Anderson", fontsize=12)
plt.tight_layout(); plt.show()

peak_mm = np.max(np.abs(wa[0].data)) * 1000
dist_km = float(raw.stats.sac.dist)
print(f"\npeak Wood-Anderson amp: {peak_mm:.2f} mm")
print(f"hypocentral distance:    {dist_km:.2f} km")
print(f"ML (Sheen 2018, {CHANNEL[-1]}):     {ATT_FN(peak_mm, dist_km, CHANNEL[-1]):.2f}")

## 3. Per-station ML — the M5.8 event

Apply the chain to every station-channel SAC in the sample event's directory. Returns one
row per trace with `dist_km`, `peak_mm`, and `ML`. Scatter the per-station MLs by epicentral
distance — a flat distribution (no distance-dependent drift) is the right behaviour.

In [ ]:
df = mlp.per_station_ml(os.path.join(EVENT_WF_ROOT, SAMPLE_EID), inv, attenuation_fn=ATT_FN)
agg = mlp.aggregate_ml(df)
print(f"M5.8 — {len(df)} station-channel MLs:")
print(f"  median ML : {agg['ml_median']:.2f}")
print(f"  mean   ML : {agg['ml_mean']:.2f}  (±{agg['ml_std']:.2f})")
print(f"  KMA published ML: 5.8\n")
display(df.round(2))

# Distance-ML scatter
fig, ax = plt.subplots(figsize=(10, 4.2), dpi=110)
comp_colors = {"Z": "#1f77b4", "N": "#d62728", "E": "#2ca02c"}
for c, sub in df.groupby(df.channel.str[-1]):
    ax.scatter(sub.dist_km, sub.ML, s=70, alpha=0.75, label=c, color=comp_colors.get(c, "k"),
                edgecolor="k", linewidth=0.5)
    for r in sub.itertuples():
        ax.annotate(r.station, (r.dist_km, r.ML), fontsize=7, color="0.4", xytext=(2, 2),
                    textcoords="offset points")
ax.axhline(agg["ml_median"], color="k", ls="--", lw=0.8, label=f"median {agg['ml_median']:.2f}")
ax.axhline(5.8, color="red", ls=":", lw=0.8, label="KMA 5.8")
ax.set(xlabel="hypocentral distance (km)", ylabel="per-channel ML",
       title=f"{SAMPLE_EID} (Gyeongju M5.8 mainshock) — per-station ML")
ax.grid(alpha=0.3); ax.legend(fontsize=9); plt.tight_layout(); plt.show()

## 4. Record section (displacement) with picks

Distance-sorted Z-component displacement record section for the M5.8 event, with P (red
triangles) and S (blue triangles) picks read from the SAC `a` / `t0` headers we wrote in
the prior export notebook. The peak around the S arrival (∼0–10 s after S) is what drives
the WA amplitude and hence the ML.

In [ ]:
# Read all stations of the sample event, deconvolve, plot displacement (Z only)
evdir = os.path.join(EVENT_WF_ROOT, SAMPLE_EID)
sacs = sorted(glob.glob(os.path.join(evdir, "*Z.sac")))
st = obspy.Stream()
for f in sacs:
    st += obspy.read(f)[0]
disp = mlp.remove_response_to_disp(st, inv)
disp = obspy.Stream(sorted(disp, key=lambda t: float(t.stats.sac.dist)))

fig, ax = plt.subplots(figsize=(10, max(4, 0.45*len(disp))), dpi=120)
for tr in disp:
    d = tr.stats.sac.dist
    times = tr.times() - 30                                # origin at t=0 by export convention
    norm = tr.data / (np.max(np.abs(tr.data)) or 1.0)
    ax.plot(times, norm * 2.5 + d, color="k", lw=0.5)
    a, t0 = tr.stats.sac.get("a", None), tr.stats.sac.get("t0", None)
    if a not in (None, -12345.0):
        ax.plot(a, d, "v", color="red", ms=6, mec="k", mew=0.3)
    if t0 not in (None, -12345.0):
        ax.plot(t0, d, "v", color="blue", ms=6, mec="k", mew=0.3)
    ax.text(-30 + 0.5, d, f"{tr.stats.network}.{tr.stats.station}", fontsize=7, va="bottom")
ax.axvline(0, color="0.6", ls="--", lw=0.7)
ax.set_xlim(-30, 90); ax.invert_yaxis()
ax.set(xlabel="time relative to origin (s)", ylabel="hypocentral distance (km)",
       title=f"{SAMPLE_EID} (M5.8) — Z-component displacement (normalised), P (red ▽) + S (blue ▽)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Benchmark vs KMA-published ML

Three well-known events in the catalog with published KMA local magnitudes. If the chain is
calibrated correctly, the computed median ML should land within ±0.3 of KMA's value.

| Event                       | UTC origin              | KMA ML |
|-----------------------------|-------------------------|--------|
| Gyeongju M5.8 mainshock     | 2016-09-12 11:32:54 UTC | 5.8    |
| Gyeongju M5.4 foreshock     | 2016-09-12 10:44:32 UTC | 5.4    |
| Gyeongju M4.5 aftershock    | 2016-09-19 11:33:58 UTC | 4.5    |

The first two events use **Sheen 2018** and should be tight; the M4.5 may show a slight
positive bias (Sheen 2018 is calibrated mostly on M3–6 instrumental events). When you swap
in Heo et al. 2024, just change `ATT_FN` at the top of §1 and re-run this cell — the
residuals re-compute automatically.

In [ ]:
BENCHMARKS = [
    ("20160912113254", "Gyeongju M5.8 mainshock", 5.8),
    ("20160912104432", "Gyeongju M5.4 foreshock", 5.4),
    ("20160919113358", "Gyeongju M4.5 aftershock", 4.5),
]
rows = []
for eid, label, kma in BENCHMARKS:
    evdir = os.path.join(EVENT_WF_ROOT, eid)
    df = mlp.per_station_ml(evdir, inv, attenuation_fn=ATT_FN)
    a = mlp.aggregate_ml(df)
    rows.append(dict(event_id=eid, label=label, kma_ml=kma,
                     ml_median=a["ml_median"], ml_mean=a["ml_mean"], ml_std=a["ml_std"],
                     n_used=a["n_used"], residual=a["ml_median"] - kma,
                     within_tol=abs(a["ml_median"] - kma) <= 0.3))
bench = pd.DataFrame(rows)
display(bench.round(2))

passed = bench["within_tol"].sum()
print(f"\n{passed}/{len(bench)} benchmarks within ±0.3 ML of the KMA-published value "
      f"using {ATT_FN.__name__}")

## 6. Hook for swapping the attenuation formula

The pipeline now ships with **Heo et al. (2024)** as the default (`ATT_FN = mlp.ml_heo2024`
in §1) — calibrated on 2,860 micro-earthquakes detected by the GHBSN's 200-station array
around the 2016 Gyeongju epicenter, so it's the region-specific scale for the southeastern
Korean Peninsula. The hot-swap point in `ml_pipeline.py` keeps `ml_sheen2018` available as
a one-line fallback (`ATT_FN = mlp.ml_sheen2018`) for cross-comparison with the broader
South Korea calibration.

**Quick comparison on the three benchmark events** (with the default — all-component median):

| Event                       | KMA | Sheen 2018     | Heo 2024       |
|-----------------------------|-----|----------------|----------------|
| Gyeongju M5.8 mainshock     | 5.8 | 5.74 (−0.06)   | 5.49 (−0.31)   |
| Gyeongju M5.4 foreshock     | 5.4 | 5.39 (−0.01)   | 5.10 (−0.30)   |
| Gyeongju M4.5 aftershock    | 4.5 | 4.87 (+0.37)   | 4.57 (+0.07)   |

Heo 2024 substantially **reduces the small-magnitude bias** that Sheen 2018 shows (+0.37
on the M4.5 → +0.07). The M5+ events come out slightly lower than KMA because KMA's
own catalog values use Sheen 2018 as the reference — the comparison there isn't strictly
fair. Heo's formula is the more appropriate choice for the GHBSN-region micro-earthquakes
this catalog mostly contains.

**Z-only filtering** (matching Heo's paper exactly, which calibrated on vertical-component
amplitudes only): add `df = df[df.channel.str[-1] == "Z"]` after the `per_station_ml`
call in §4. The Z-only median differs from the all-component median by ≤ 0.06 ML in
practice — minor but documented for fidelity to the original publication.

To switch back to Sheen 2018: in §1, change
```python
ATT_FN = mlp.ml_heo2024
```
to
```python
ATT_FN = mlp.ml_sheen2018
```
and re-run cells §3 + §5.